In [1]:
import cv2
import numpy as np
import torch
import os
import sys
import time
from datetime import datetime
import matplotlib.pyplot as plt
import math
import gc # For garbage collection

# --- Add project root to Python path ---
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)
print(f"Project Root: {project_root}")

# --- Your project's imports ---
from config import (MODEL_PATH, MODEL_INPUT_SIZE, CONF_THRESHOLD, IOU_THRESHOLD, TARGET_FPS, PYVIDEOREADER_DECODE_SEGMENT_SIZE)
from models.model_loader import load_model, get_device
from core.image_processing import enhance_frame_for_detection
from core.video_processor import VideoProcessor

# --- Attempt to import PyVideoReader ---
PYVIDEOREADER_AVAILABLE = False
PyVideoReader = None
try:
    from video_reader import PyVideoReader
    PYVIDEOREADER_AVAILABLE = True
    print("PyVideoReader (video_reader-rs) found and available.")
except ImportError:
    print("PyVideoReader (video_reader-rs) not found. This benchmark will fall back to OpenCV if PVR is selected.")

# --- Helper Functions ---
def display_image_jupyter(image_bgr, title="Image"):
    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(10, 8))
    plt.imshow(image_rgb)
    plt.title(title)
    plt.axis('off')
    plt.show()

print("Setup Complete. Necessary modules imported.")

Project Root: c:\Users\swest\Desktop\traffic-capturing\traffic-reporting
--- Configuration Loaded (Tuned for Exit Accuracy) ---
Line Definition Mode: HARDCODED
Input Video: ['C:\\Users\\swest\\Desktop\\traffic-capturing\\sample_videos\\5 minute test - 4 Way Intersection.mp4']
Start Date/Time: 250604 / 003142108
Using Model: C:\Users\swest\Desktop\traffic-capturing\traffic-reporting\models\weights\silent_contract_9082_original_640px_bs416.engine (Conf: 0.05, IoU: 0.6)
Model Input Size: 640x640
Using Hardcoded Lines (config.py)
Target FPS: 25, Batch Size: 416, Threads: 48
Mixed Precision: True, CUDA Streams: 32
Detailed Perf Metrics: True, Debug Mode: False
--------------------------------------------------------------------


PyVideoReader (video_reader-rs) found and available.
PyVideoReader (video_reader-rs) found and available.
Setup Complete. Necessary modules imported.


In [2]:
# --- Configuration for this Benchmark Run ---

VIDEO_PATH = r"C:\Users\swest\Desktop\traffic-capturing\sample_videos\5 min test - Large FOV.mp4"
OUTPUT_DIR_BASE = r"C:\Users\swest\Desktop\traffic-capturing\traffic-reporting\utility_scripts\benchmark_detection_outputs"

# --- Configuration for this Benchmark Run ---

FRAME_INTERVAL_CONFIG = 25
CONF_THRESHOLD_OVERRIDE = None
ENHANCEMENT_LEVEL = "none"
MAX_FRAMES_TO_PROCESS = None
DISPLAY_INTERACTIVE_FRAMES = None
SAVE_ANNOTATED_FRAMES = True
USE_PYVIDEOREADER = True and PYVIDEOREADER_AVAILABLE

timestamp_run = datetime.now().strftime("%Y%m%d_%H%M")
if os.path.basename(VIDEO_PATH) != "benchmark_video.mp4" and os.path.exists(VIDEO_PATH):
    video_name_stem = os.path.splitext(os.path.basename(VIDEO_PATH))[0]
    OUTPUT_DIR_RUN = os.path.join(OUTPUT_DIR_BASE, f"{video_name_stem}_{timestamp_run}")
else:
    OUTPUT_DIR_RUN = os.path.join(OUTPUT_DIR_BASE, f"run_{timestamp_run}")

if SAVE_ANNOTATED_FRAMES:
    os.makedirs(OUTPUT_DIR_RUN, exist_ok=True)
    print(f"Annotated frames will be saved to: {OUTPUT_DIR_RUN}")
else:
    print("Annotated frames will NOT be saved to disk.")

if not os.path.exists(VIDEO_PATH):
    print(f"ERROR: Video file not found at '{VIDEO_PATH}'. Please set the VIDEO_PATH variable correctly.")
    raise FileNotFoundError(f"Video not found: {VIDEO_PATH}")
else:
    print(f"Using video: {VIDEO_PATH}")

if USE_PYVIDEOREADER:
    print("Attempting to use PyVideoReader.")
else:
    print("Using OpenCV for video reading.")

Annotated frames will be saved to: C:\Users\swest\Desktop\traffic-capturing\traffic-reporting\utility_scripts\benchmark_detection_outputs\5 min test - Large FOV_20250604_0031
Using video: C:\Users\swest\Desktop\traffic-capturing\sample_videos\5 min test - Large FOV.mp4
Attempting to use PyVideoReader.


In [3]:
print("Initializing VideoProcessor instance for preprocessing...")
vp = VideoProcessor()

print("\nLoading model (external to VideoProcessor instance for this script)...")
device = get_device() # Use global get_device for this script
effective_conf_thresh = CONF_THRESHOLD_OVERRIDE if CONF_THRESHOLD_OVERRIDE is not None else CONF_THRESHOLD
# MODEL_PATH = r'C:\Users\swest\Desktop\traffic-capturing\traffic-reporting\utility_scripts\yolov8l.pt'
model = load_model(MODEL_PATH, device, effective_conf_thresh, IOU_THRESHOLD) # load_model takes device
vp.device = device # Ensure VideoProcessor instance uses the same device as the loaded model

print(f"Model loaded on device: {device}")

class_names_map = vp.class_id_to_name_map
# class_names_map = {}
default_class_label = 'unknown'

if not class_names_map:
    print(f"CRITICAL from VideoProcessor: Class name map is empty. Using generic names '{default_class_label}_ID'.")
    # Fallback if vp didn't load them for some reason (should not happen if YAML is correct)
    if hasattr(model, 'names') and model.names:
        class_names_map = model.names
        print(f"Fell back to model.names: {class_names_map}")
    else:
        class_names_map = {i: f"{default_class_label}_{i}" for i in range(80)} # Assume up to 80
        print(f"Fell back to very generic names. First few: { {k:v for i,(k,v) in enumerate(class_names_map.items()) if i<3} }")

print(f"Using class name map (from VideoProcessor instance): {class_names_map if class_names_map else 'EMPTY - CHECK VP INIT'}")

Initializing VideoProcessor instance for preprocessing...
Using GPU: NVIDIA GeForce RTX 3090
Configuration: Advanced video reader (PyVideoReader) will be used.

Loading model (external to VideoProcessor instance for this script)...
Using GPU: NVIDIA GeForce RTX 3090
Model loaded on device: cuda
Using class name map (from VideoProcessor instance): {0: 'Bus', 1: 'Cyclist', 2: 'Light Vehicle', 3: 'Long Truck', 4: 'Medium Truck', 5: 'Minibus Taxi', 6: 'Motorcycle', 7: 'Pedestrian', 8: 'Short Truck'}


Benchmark - Standard Processing

In [4]:
print(f"\n--- Starting Benchmark Processing ---")

total_video_frames = 0
original_video_fps = 0.0
_cap_cv_check = cv2.VideoCapture(VIDEO_PATH)
if _cap_cv_check.isOpened():
    total_video_frames = int(_cap_cv_check.get(cv2.CAP_PROP_FRAME_COUNT))
    original_video_fps = _cap_cv_check.get(cv2.CAP_PROP_FPS)
    _cap_cv_check.release()
else:
    print(f"Error: OpenCV could not open video {VIDEO_PATH} to get properties.")
    raise IOError(f"Failed to get video properties for {VIDEO_PATH}")

if total_video_frames == 0 or original_video_fps == 0:
    print("Warning: Could not reliably get video properties. Using defaults.")
    total_video_frames = 1000000
    original_video_fps = TARGET_FPS if TARGET_FPS > 0 else 25.0

print(f"Video Properties: ~{total_video_frames} frames, {original_video_fps:.2f} FPS (Original)")
frame_skip_interval = FRAME_INTERVAL_CONFIG

frames_actually_processed = 0
total_preprocessing_time_ms = 0
total_inference_time_ms = 0
total_enhancement_time_ms = 0
raw_frame_index_overall = 0

cap_for_originals = cv2.VideoCapture(VIDEO_PATH)
if not cap_for_originals.isOpened():
    print(f"CRITICAL ERROR: OpenCV could not open video {VIDEO_PATH} for original frames.")
    raise IOError(f"Failed to open video for original frames: {VIDEO_PATH}")

current_batch_for_preprocessing = [] # To hold (frame_content, metadata) tuples for _preprocess_batch

if USE_PYVIDEOREADER:
    pvr_instance = None
    try:
        print(f"PVR: Initializing for {VIDEO_PATH}, target: {MODEL_INPUT_SIZE}x{MODEL_INPUT_SIZE}")
        pvr_instance = PyVideoReader(VIDEO_PATH, device='cpu', # PVR gives CPU numpy array
                                     resize_shorter_side=MODEL_INPUT_SIZE,
                                     resize_longer_side=MODEL_INPUT_SIZE)
        
        num_segments = math.ceil(total_video_frames / PYVIDEOREADER_DECODE_SEGMENT_SIZE)
        print(f"PVR: Segments={num_segments}")

        for seg_idx in range(num_segments):
            if MAX_FRAMES_TO_PROCESS is not None and frames_actually_processed >= MAX_FRAMES_TO_PROCESS: break
            segment_start_raw_idx = seg_idx * PYVIDEOREADER_DECODE_SEGMENT_SIZE
            segment_end_raw_idx = min((seg_idx + 1) * PYVIDEOREADER_DECODE_SEGMENT_SIZE, total_video_frames)
            if segment_start_raw_idx >= segment_end_raw_idx: continue
            
            segment_frames_np_rgb = pvr_instance.decode(start_frame=segment_start_raw_idx, end_frame=segment_end_raw_idx)
            if segment_frames_np_rgb is None or segment_frames_np_rgb.shape[0] == 0: continue

            for frame_in_segment_idx in range(segment_frames_np_rgb.shape[0]):
                current_original_video_frame_idx = segment_start_raw_idx + frame_in_segment_idx
                raw_frame_index_overall = current_original_video_frame_idx

                if current_original_video_frame_idx % frame_skip_interval != 0: continue

                # Get original BGR frame for drawing
                original_frame_bgr = None
                cap_for_originals.set(cv2.CAP_PROP_POS_FRAMES, current_original_video_frame_idx)
                ret_orig, original_frame_bgr = cap_for_originals.read()
                if not ret_orig: continue

                frames_actually_processed += 1
                print(f"PVR Processing Original Frame #{current_original_video_frame_idx} (Actual Proc. #{frames_actually_processed})")
                
                frame_content_for_model = segment_frames_np_rgb[frame_in_segment_idx] # RGB, HWC, Model-Sized

                if ENHANCEMENT_LEVEL and ENHANCEMENT_LEVEL != 'none':
                    start_enhance_t = time.monotonic()
                    # enhance_frame_for_detection expects RGB, HWC numpy array
                    frame_content_for_model = enhance_frame_for_detection(frame_content_for_model, ENHANCEMENT_LEVEL)
                    enhancement_t_ms = (time.monotonic() - start_enhance_t) * 1000
                    total_enhancement_time_ms += enhancement_t_ms
                    print(f"  Enhancement time: {enhancement_t_ms:.2f} ms")
                
                # Prepare for vp._preprocess_batch
                # Metadata: PVR provides RGB, model-sized frame
                metadata = {'is_rgb': True, 'source': 'pvr_benchmark'}
                current_batch_for_preprocessing.append((frame_content_for_model, metadata))
                
                # For benchmark, process one frame at a time with _preprocess_batch
                start_preprocess_t = time.monotonic()
                # vp._preprocess_batch expects list of (frame, metadata)
                # vp.device should be set to the inference device
                preprocessed_tensor = vp._preprocess_batch(current_batch_for_preprocessing)
                current_batch_for_preprocessing.clear() # Clear after use
                preprocessing_t_ms = (time.monotonic() - start_preprocess_t) * 1000
                total_preprocessing_time_ms += preprocessing_t_ms
                print(f"  Preprocessing time (via vp._preprocess_batch): {preprocessing_t_ms:.2f} ms")

                if preprocessed_tensor is None:
                    print("  Skipping frame: Preprocessing returned None.")
                    continue

                # Inference (same as before)
                start_inference_t = time.monotonic()
                with torch.no_grad(): results = model(preprocessed_tensor, verbose=False)
                inference_t_ms = (time.monotonic() - start_inference_t) * 1000
                total_inference_time_ms += inference_t_ms
                print(f"  Inference time: {inference_t_ms:.2f} ms")

                # Postprocessing & Drawing (same as before, on original_frame_bgr)
                annotated_display_frame = original_frame_bgr.copy()
                result_obj = results[0]
                num_detections_on_frame = 0
                if hasattr(result_obj, 'boxes') and result_obj.boxes is not None:
                    for box_data in result_obj.boxes.data:
                        box_cpu = box_data.cpu().numpy(); x1m,y1m,x2m,y2m,conf,cls_id_f = box_cpu
                        cls_id = int(cls_id_f); num_detections_on_frame +=1
                        label_str = class_names_map.get(cls_id, f"{default_class_label}_{cls_id}")
                        oh, ow = annotated_display_frame.shape[:2]
                        x1,y1,x2,y2 = int(x1m*ow/MODEL_INPUT_SIZE),int(y1m*oh/MODEL_INPUT_SIZE),int(x2m*ow/MODEL_INPUT_SIZE),int(y2m*oh/MODEL_INPUT_SIZE)
                        cv2.rectangle(annotated_display_frame,(x1,y1),(x2,y2),(0,255,0),2)
                        cv2.putText(annotated_display_frame,f"{label_str}:{conf:.2f}",(x1,y1-10 if y1>20 else y1+20),cv2.FONT_HERSHEY_SIMPLEX,0.6,(0,255,0),2)
                    print(f"  Detections: {num_detections_on_frame}")
                else: print("  No detections.")

                if SAVE_ANNOTATED_FRAMES:
                    cv2.imwrite(os.path.join(OUTPUT_DIR_RUN, f"frame_{current_original_video_frame_idx:06d}_pvr.jpg"), annotated_display_frame)
                if DISPLAY_INTERACTIVE_FRAMES:
                    display_image_jupyter(annotated_display_frame, title=f"PVR - Ann. Frame #{current_original_video_frame_idx}")
                
                if MAX_FRAMES_TO_PROCESS is not None and frames_actually_processed >= MAX_FRAMES_TO_PROCESS: break
            del segment_frames_np_rgb; gc.collect()
            if MAX_FRAMES_TO_PROCESS is not None and frames_actually_processed >= MAX_FRAMES_TO_PROCESS: break
    except Exception as e_pvr: print(f"PVR Error: {e_pvr}"); import traceback; traceback.print_exc()
    finally:
        if pvr_instance: del pvr_instance; gc.collect()

else: # OpenCV Fallback
    print("Using OpenCV video reader for benchmark.")
    cap_cv = cv2.VideoCapture(VIDEO_PATH)
    if not cap_cv.isOpened(): print(f"Error: OpenCV could not open {VIDEO_PATH}")
    else:
        while True:
            ret, frame_bgr_original = cap_cv.read()
            if not ret: break
            raw_frame_index_overall += 1
            if raw_frame_index_overall % frame_skip_interval != 0: continue

            # Get original BGR frame (it's already frame_bgr_original from cap_cv.read())
            if frame_bgr_original is None: continue # Should not happen if ret is True

            frames_actually_processed += 1
            print(f"OpenCV Processing Original Frame #{raw_frame_index_overall} (Actual Proc. #{frames_actually_processed})")

            frame_content_for_model = frame_bgr_original.copy() # BGR, Original Size

            if ENHANCEMENT_LEVEL and ENHANCEMENT_LEVEL != 'none':
                start_enhance_t = time.monotonic()
                # enhance_frame_for_detection expects RGB.
                frame_rgb_enh = cv2.cvtColor(frame_content_for_model, cv2.COLOR_BGR2RGB)
                enhanced_rgb = enhance_frame_for_detection(frame_rgb_enh, ENHANCEMENT_LEVEL)
                # For _preprocess_batch with 'opencv' source, it expects BGR.
                frame_content_for_model = cv2.cvtColor(enhanced_rgb, cv2.COLOR_RGB2BGR)
                enhancement_t_ms = (time.monotonic() - start_enhance_t) * 1000
                total_enhancement_time_ms += enhancement_t_ms
                print(f"  Enhancement time: {enhancement_t_ms:.2f} ms")

            # Prepare for vp._preprocess_batch
            # Metadata: OpenCV provides BGR, original size frame
            metadata = {'is_rgb': False, 'source': 'opencv_benchmark'}
            current_batch_for_preprocessing.append((frame_content_for_model, metadata))

            start_preprocess_t = time.monotonic()
            preprocessed_tensor = vp._preprocess_batch(current_batch_for_preprocessing)
            current_batch_for_preprocessing.clear()
            preprocessing_t_ms = (time.monotonic() - start_preprocess_t) * 1000
            total_preprocessing_time_ms += preprocessing_t_ms
            print(f"  Preprocessing time (via vp._preprocess_batch): {preprocessing_t_ms:.2f} ms")

            if preprocessed_tensor is None:
                print("  Skipping frame: Preprocessing returned None.")
                continue
            
            # Inference, Postprocessing, Drawing, Saving (same as PVR path from here on)
            start_inference_t = time.monotonic()
            with torch.no_grad(): results = model(preprocessed_tensor, verbose=False)
            inference_t_ms = (time.monotonic() - start_inference_t) * 1000
            total_inference_time_ms += inference_t_ms
            print(f"  Inference time: {inference_t_ms:.2f} ms")

            annotated_display_frame = frame_bgr_original.copy() # Draw on the frame read by cap_cv
            result_obj = results[0]
            num_detections_on_frame = 0
            if hasattr(result_obj, 'boxes') and result_obj.boxes is not None:
                for box_data in result_obj.boxes.data:
                    box_cpu = box_data.cpu().numpy(); x1m,y1m,x2m,y2m,conf,cls_id_f = box_cpu
                    cls_id = int(cls_id_f); num_detections_on_frame +=1
                    label_str = class_names_map.get(cls_id, f"{default_class_label}_{cls_id}")
                    oh, ow = annotated_display_frame.shape[:2]
                    x1,y1,x2,y2 = int(x1m*ow/MODEL_INPUT_SIZE),int(y1m*oh/MODEL_INPUT_SIZE),int(x2m*ow/MODEL_INPUT_SIZE),int(y2m*oh/MODEL_INPUT_SIZE)
                    cv2.rectangle(annotated_display_frame,(x1,y1),(x2,y2),(0,255,0),2)
                    cv2.putText(annotated_display_frame,f"{label_str}:{conf:.2f}",(x1,y1-10 if y1>20 else y1+20),cv2.FONT_HERSHEY_SIMPLEX,0.6,(0,255,0),2)
                print(f"  Detections: {num_detections_on_frame}")
            else: print("  No detections.")

            if SAVE_ANNOTATED_FRAMES:
                cv2.imwrite(os.path.join(OUTPUT_DIR_RUN, f"frame_{raw_frame_index_overall:06d}_cv.jpg"), annotated_display_frame)
            if DISPLAY_INTERACTIVE_FRAMES:
                display_image_jupyter(annotated_display_frame, title=f"OpenCV - Ann. Frame #{raw_frame_index_overall}")
            
            if MAX_FRAMES_TO_PROCESS is not None and frames_actually_processed >= MAX_FRAMES_TO_PROCESS: break
        if cap_cv.isOpened(): cap_cv.release()

if cap_for_originals.isOpened(): cap_for_originals.release()
gc.collect()

# --- Final Summary (same as before) ---
print("\n--- Benchmark Run Summary ---")
if frames_actually_processed > 0:
    avg_enhance_ms = (total_enhancement_time_ms / frames_actually_processed) if ENHANCEMENT_LEVEL != 'none' and total_enhancement_time_ms > 0 else 0
    avg_preprocess_ms = total_preprocessing_time_ms / frames_actually_processed
    avg_inference_ms = total_inference_time_ms / frames_actually_processed
    print(f"Total Video Frames (approx): {total_video_frames}")
    print(f"Frames Processed: {frames_actually_processed}")
    if ENHANCEMENT_LEVEL != 'none': print(f"Avg Enhancement Time: {avg_enhance_ms:.2f} ms/frame")
    print(f"Avg Preprocessing Time: {avg_preprocess_ms:.2f} ms/frame")
    print(f"Avg Inference Time: {avg_inference_ms:.2f} ms/frame")
    total_avg_ms = avg_preprocess_ms + avg_inference_ms + avg_enhance_ms
    print(f"Avg Total (Enh+PreP+Inf): {total_avg_ms:.2f} ms/frame")
    if total_avg_ms > 0 : print(f"Equivalent FPS (processed): {1000 / total_avg_ms:.2f} FPS")
else: print("No frames processed.")
if SAVE_ANNOTATED_FRAMES and os.path.exists(OUTPUT_DIR_RUN) and len(os.listdir(OUTPUT_DIR_RUN)) > 0 :
    print(f"Annotated frames saved to: {OUTPUT_DIR_RUN}")
elif SAVE_ANNOTATED_FRAMES: print(f"SAVE_ANNOTATED_FRAMES was True, but no frames saved to {OUTPUT_DIR_RUN}.")
print("Notebook execution finished.")


--- Starting Benchmark Processing ---
Video Properties: ~7500 frames, 25.00 FPS (Original)
PVR: Initializing for C:\Users\swest\Desktop\traffic-capturing\sample_videos\5 min test - Large FOV.mp4, target: 640x640
PVR: Segments=1
PVR Processing Original Frame #0 (Actual Proc. #1)
  Preprocessing time (via vp._preprocess_batch): 31.00 ms
Loading C:\Users\swest\Desktop\traffic-capturing\traffic-reporting\models\weights\silent_contract_9082_original_640px_bs416.engine for TensorRT inference...
WARNING Metadata not found for 'model=C:\Users\swest\Desktop\traffic-capturing\traffic-reporting\models\weights\silent_contract_9082_original_640px_bs416.engine'
  Inference time: 1328.00 ms
  Detections: 8
PVR Processing Original Frame #25 (Actual Proc. #2)
  Preprocessing time (via vp._preprocess_batch): 16.00 ms
  Inference time: 0.00 ms
  Detections: 7
PVR Processing Original Frame #50 (Actual Proc. #3)
  Preprocessing time (via vp._preprocess_batch): 0.00 ms
  Inference time: 0.00 ms
  Detection

------------------------------------------------

-----------------------------------

Using SAHI

In [5]:
SAHI_SLICE_HEIGHT = 480
SAHI_SLICE_WIDTH = 640
SAHI_OVERLAP_HEIGHT_RATIO = 0.2
SAHI_OVERLAP_WIDTH_RATIO = 0.2
SAHI_POSTPROCESS_CONF_THRESHOLD = 0.5 # You might want a slightly different threshold for SAHI
SAHI_POSTPROCESS_IOU_THRESHOLD = 0.5
print(f"  SAHI Params: Slice HxW={SAHI_SLICE_HEIGHT}x{SAHI_SLICE_WIDTH}, Overlap={SAHI_OVERLAP_HEIGHT_RATIO}, Post-Conf={SAHI_POSTPROCESS_CONF_THRESHOLD}")

# --- Optional: Base Model Confidence for SAHI Slices ---
# This confidence is used by the base YOLO model when predicting on individual slices.
# SAHI_POSTPROCESS_CONF_THRESHOLD is applied *after* slices are merged.
# Setting this lower than SAHI_POSTPROCESS_CONF_THRESHOLD can sometimes help detect faint objects on slices.
# This would affect `effective_conf_thresh_for_base_model` in Cell 3.
BASE_MODEL_CONF_FOR_SAHI_SLICES = 0.1 # Example: lower threshold for slice predictions
if CONF_THRESHOLD_OVERRIDE is None or CONF_THRESHOLD_OVERRIDE > BASE_MODEL_CONF_FOR_SAHI_SLICES :
    CONF_THRESHOLD_OVERRIDE = BASE_MODEL_CONF_FOR_SAHI_SLICES # This will be picked up in Cell 3
    print(f"  Override: CONF_THRESHOLD_OVERRIDE (for base model on slices) set to {CONF_THRESHOLD_OVERRIDE}")

sahi_str_override = "_sahi"
reader_str_override = "_pvr" if USE_PYVIDEOREADER and PYVIDEOREADER_AVAILABLE else "_cv"

if os.path.exists(VIDEO_PATH): # Check VIDEO_PATH from Cell 2
    video_name_stem_override = os.path.splitext(os.path.basename(VIDEO_PATH))[0]
else:
    video_name_stem_override = "video_not_found" # Should have been caught in Cell 2

# Re-generate OUTPUT_DIR_RUN with SAHI flag if SAHI is active
OUTPUT_DIR_RUN = os.path.join(OUTPUT_DIR_BASE, f"{video_name_stem_override}{reader_str_override}{sahi_str_override}_{timestamp_run}") # timestamp_run from Cell 2

if SAVE_ANNOTATED_FRAMES: # SAVE_ANNOTATED_FRAMES from Cell 2
    os.makedirs(OUTPUT_DIR_RUN, exist_ok=True) # Ensure it's created
    print(f"  Override: Output directory updated for SAHI run: {OUTPUT_DIR_RUN}")


print("SAHI Specific Overrides applied (if SAHI enabled and available).")
print("-" * 50)

  SAHI Params: Slice HxW=480x640, Overlap=0.2, Post-Conf=0.5
  Override: CONF_THRESHOLD_OVERRIDE (for base model on slices) set to 0.1
  Override: Output directory updated for SAHI run: C:\Users\swest\Desktop\traffic-capturing\traffic-reporting\utility_scripts\benchmark_detection_outputs\5 min test - Large FOV_pvr_sahi_20250604_0031
SAHI Specific Overrides applied (if SAHI enabled and available).
--------------------------------------------------


In [6]:
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction

sahi_detection_model = None

try:
    # The confidence_threshold for AutoDetectionModel is for the base model's predictions on slices.
    # _effective_conf_for_base_load (which is CONF_THRESHOLD_OVERRIDE if set by Cell 2.5) is appropriate here.
    sahi_detection_model = AutoDetectionModel.from_pretrained(
        model_type='yolov8',
        model_path=r'C:\Users\swest\Desktop\traffic-capturing\traffic-reporting\utility_scripts\yolov8l.pt', # SAHI's YOLOv8 handler can often use path.
                                # Alternatively, could try `model=model` if path fails.
        confidence_threshold=BASE_MODEL_CONF_FOR_SAHI_SLICES,
        device=device,
        # image_size=MODEL_INPUT_SIZE # Optional for SAHI YOLOv8 handler
    )

    # Attempt to propagate names to SAHI's internal model representation.
    # This is a "best effort" as SAHI's handling can vary.
    # The `class_names_map_for_annotation` below is the primary source for drawing.
    if hasattr(model, 'names') and model.names and isinstance(model.names, dict):
        try:
            if hasattr(sahi_detection_model, 'model') and hasattr(sahi_detection_model.model, 'names'):
                sahi_detection_model.model.names = model.names
            # SAHI also uses these for its internal category management
            sahi_detection_model.category_mapping = {k: v for k, v in model.names.items()}
            sahi_detection_model.category_names = list(model.names.values())
            print("  Attempted to propagate names to SAHI detection model.")
            if hasattr(sahi_detection_model.model, 'names'):
                    print(f"    SAHI internal model names now: {sahi_detection_model.model.names}")
        except Exception as e_sahi_names:
            print(f"  Note: Could not directly set/propagate names on SAHI model object: {e_sahi_names}")
    
    print("SAHI DetectionModel setup complete.")

except Exception as e_sahi_setup:
    print(f"ERROR setting up SAHI DetectionModel: {e_sahi_setup}")


# --- Finalize Class Names Map for Annotation ---
# This map will be used in Cell 4 for drawing labels, ensuring consistency.
# It prioritizes the names loaded by the VideoProcessor instance (from your YAML).
class_names_map_for_annotation = vp.class_id_to_name_map
class_names_map_for_annotation = {}
default_class_label_for_annotation = 'unknown'

if not class_names_map_for_annotation: # If vp's map is empty for some reason
    print(f"WARNING: Class name map from VideoProcessor (vp.class_id_to_name_map) is empty.")
    if hasattr(model, 'names') and model.names and isinstance(model.names, dict):
        class_names_map_for_annotation = model.names
        print(f"  Fell back to using model.names for annotation: {class_names_map_for_annotation}")
    else:
        # Absolute fallback if no names can be found anywhere
        class_names_map_for_annotation = {i: f"{default_class_label_for_annotation}_{i}" for i in range(80)} # Default to 80 COCO-like generic names
        print(f"  CRITICAL FALLBACK: Using very generic names for annotation (e.g., '{default_class_label_for_annotation}_0')")

print(f"\nFinal class name map to be used for annotations in Cell 4: "
      f"{class_names_map_for_annotation if class_names_map_for_annotation else 'ERROR - MAP IS EMPTY'}")


  Note: Could not directly set/propagate names on SAHI model object: can't set attribute 'names'
SAHI DetectionModel setup complete.
  Fell back to using model.names for annotation: {0: 'class0', 1: 'class1', 2: 'class2', 3: 'class3', 4: 'class4', 5: 'class5', 6: 'class6', 7: 'class7', 8: 'class8', 9: 'class9', 10: 'class10', 11: 'class11', 12: 'class12', 13: 'class13', 14: 'class14', 15: 'class15', 16: 'class16', 17: 'class17', 18: 'class18', 19: 'class19', 20: 'class20', 21: 'class21', 22: 'class22', 23: 'class23', 24: 'class24', 25: 'class25', 26: 'class26', 27: 'class27', 28: 'class28', 29: 'class29', 30: 'class30', 31: 'class31', 32: 'class32', 33: 'class33', 34: 'class34', 35: 'class35', 36: 'class36', 37: 'class37', 38: 'class38', 39: 'class39', 40: 'class40', 41: 'class41', 42: 'class42', 43: 'class43', 44: 'class44', 45: 'class45', 46: 'class46', 47: 'class47', 48: 'class48', 49: 'class49', 50: 'class50', 51: 'class51', 52: 'class52', 53: 'class53', 54: 'class54', 55: 'class55

In [7]:
print(f"\n--- Starting Benchmark Processing ---")

# --- Get Video Properties (Common for both readers) ---
total_video_frames = 0
original_video_fps = 0.0
_cap_cv_check = cv2.VideoCapture(VIDEO_PATH)
if _cap_cv_check.isOpened():
    total_video_frames = int(_cap_cv_check.get(cv2.CAP_PROP_FRAME_COUNT))
    original_video_fps = _cap_cv_check.get(cv2.CAP_PROP_FPS)
    _cap_cv_check.release()
else:
    print(f"Error: OpenCV could not open video {VIDEO_PATH} to get properties.")
    raise IOError(f"Failed to get video properties for {VIDEO_PATH}")

if total_video_frames == 0 or original_video_fps == 0:
    print("Warning: Could not reliably get video properties. Using defaults.")
    total_video_frames = 1000000 # Large number
    original_video_fps = TARGET_FPS if TARGET_FPS > 0 else 25.0

print(f"Video Properties: ~{total_video_frames} frames, {original_video_fps:.2f} FPS (Original)")
frame_skip_interval = FRAME_INTERVAL_CONFIG

# --- Initialize Counters ---
frames_actually_processed = 0
total_preprocessing_time_ms_std = 0 # Specific to standard preprocessing
total_inference_time_ms = 0 # Will hold SAHI time or (Std_PreP + Std_Inf) time
total_enhancement_time_ms = 0
raw_frame_index_overall = 0 # Tracks overall frames *read* from the source video

# --- OpenCV VideoCapture for getting original frames for annotation ---
cap_for_originals = cv2.VideoCapture(VIDEO_PATH)
if not cap_for_originals.isOpened():
    print(f"CRITICAL ERROR: OpenCV could not open video {VIDEO_PATH} for original frames.")
    raise IOError(f"Failed to open video for original frames: {VIDEO_PATH}")

# --- Determine effective model/runner ---
# model_obj_for_sahi is the base YOLO model loaded in Cell 3
# sahi_detection_model is the SAHI wrapper loaded in Cell 3 if USE_SAHI
# vp is the VideoProcessor instance from Cell 3 for _preprocess_batch and class names

# --- Video Reading Loop Setup ---
reader_frame_source = None
segment_iterator = range(1) # Default for OpenCV (runs main loop once)
pvr_segment_details = {} # To store PVR specific segment info if used

if USE_PYVIDEOREADER:
    print(f"PVR: Initializing for {VIDEO_PATH}, target: {MODEL_INPUT_SIZE}x{MODEL_INPUT_SIZE}")
    reader_frame_source = PyVideoReader(VIDEO_PATH, device='cpu',
                                        resize_shorter_side=MODEL_INPUT_SIZE,
                                        resize_longer_side=MODEL_INPUT_SIZE)
    pvr_segment_details['num_segments'] = math.ceil(total_video_frames / PYVIDEOREADER_DECODE_SEGMENT_SIZE)
    segment_iterator = range(pvr_segment_details['num_segments']) # Iterate through actual segments
    print(f"PVR Source: Segments={pvr_segment_details['num_segments']}")
else: # OpenCV
    print("OpenCV Source")
    reader_frame_source = cv2.VideoCapture(VIDEO_PATH)
    if not reader_frame_source.isOpened():
        raise IOError(f"OpenCV reader failed for {VIDEO_PATH}")

# --- Main Loop ---
for seg_idx in segment_iterator: # This loop is for PVR segments, or runs once for OpenCV
    if MAX_FRAMES_TO_PROCESS is not None and frames_actually_processed >= MAX_FRAMES_TO_PROCESS:
        break

    current_pvr_segment_frames_rgb = None # Holds decoded frames if PVR is used for this segment
    pvr_current_segment_start_idx = 0

    if USE_PYVIDEOREADER:
        pvr_current_segment_start_idx = seg_idx * PYVIDEOREADER_DECODE_SEGMENT_SIZE
        segment_end_raw_idx = min((seg_idx + 1) * PYVIDEOREADER_DECODE_SEGMENT_SIZE, total_video_frames)
        if pvr_current_segment_start_idx >= segment_end_raw_idx:
            continue
        
        print(f"\nPVR: Decoding segment {seg_idx+1}/{pvr_segment_details['num_segments']} "
              f"(raw frames {pvr_current_segment_start_idx} to {segment_end_raw_idx-1})")
        current_pvr_segment_frames_rgb = reader_frame_source.decode(start_frame=pvr_current_segment_start_idx,
                                                                    end_frame=segment_end_raw_idx)
        if current_pvr_segment_frames_rgb is None or current_pvr_segment_frames_rgb.shape[0] == 0:
            print(f"PVR: Segment {seg_idx+1} returned no frames.")
            continue
        print(f"PVR: Decoded {current_pvr_segment_frames_rgb.shape[0]} frames from segment.")
    
    # Inner loop: Iterates through frames from PVR segment, or reads frame by frame for OpenCV
    frame_idx_in_current_pvr_segment = -1 # For PVR segment indexing

    while True: # This loop processes one logical frame from the video
        if MAX_FRAMES_TO_PROCESS is not None and frames_actually_processed >= MAX_FRAMES_TO_PROCESS:
            break
        
        # --- Get frame content for model and its original index ---
        frame_content_for_model_input = None # This will be fed to model/SAHI (might be RGB or BGR)
        current_original_video_frame_idx = -1
        is_input_rgb = False # Is frame_content_for_model_input RGB?

        if USE_PYVIDEOREADER:
            frame_idx_in_current_pvr_segment += 1
            if frame_idx_in_current_pvr_segment >= current_pvr_segment_frames_rgb.shape[0]:
                break # End of this PVR segment
            current_original_video_frame_idx = pvr_current_segment_start_idx + frame_idx_in_current_pvr_segment
            # PVR gives RGB, HWC, Model-Sized frame. This is NOT directly passed to SAHI.
            # SAHI needs the original BGR. We use PVR's output only if NOT using SAHI and want pre-resized input.
            # For SAHI, we'll use original_frame_bgr obtained via cap_for_originals.
            # For standard non-SAHI inference with PVR, this frame can be used.
            _pvr_frame_rgb_model_sized = current_pvr_segment_frames_rgb[frame_idx_in_current_pvr_segment]
            is_input_rgb = True 
        else: # OpenCV
            ret_cv, frame_bgr_original_size = reader_frame_source.read()
            if not ret_cv:
                break # End of video for OpenCV
            current_original_video_frame_idx = int(reader_frame_source.get(cv2.CAP_PROP_POS_FRAMES)) - 1
            # OpenCV gives BGR, HWC, Original-Sized. SAHI can take this directly.
            # Standard non-SAHI inference will also start from this.
            is_input_rgb = False

        raw_frame_index_overall = current_original_video_frame_idx

        if current_original_video_frame_idx % frame_skip_interval != 0:
            continue

        # Get original BGR frame for drawing (always needed) and for SAHI/Std-CV input
        original_frame_bgr = None
        cap_for_originals.set(cv2.CAP_PROP_POS_FRAMES, current_original_video_frame_idx)
        ret_orig, original_frame_bgr = cap_for_originals.read()
        if not ret_orig or original_frame_bgr is None:
            print(f"Warning: OpenCV could not read original frame at index {current_original_video_frame_idx} for drawing.")
            continue

        frames_actually_processed += 1
        print(f"Processing Original Frame #{current_original_video_frame_idx} (Actual Processed Frame #{frames_actually_processed})")

        # --- Image for Inference (BGR, HWC, Original Size, possibly enhanced) ---
        image_to_infer_on_bgr = original_frame_bgr.copy()

        if ENHANCEMENT_LEVEL and ENHANCEMENT_LEVEL != 'none':
            start_enhance_t = time.monotonic()
            img_rgb_to_enhance = cv2.cvtColor(image_to_infer_on_bgr, cv2.COLOR_BGR2RGB)
            enhanced_rgb = enhance_frame_for_detection(img_rgb_to_enhance, ENHANCEMENT_LEVEL)
            image_to_infer_on_bgr = cv2.cvtColor(enhanced_rgb, cv2.COLOR_RGB2BGR) # Back to BGR
            enhancement_t_ms = (time.monotonic() - start_enhance_t) * 1000
            total_enhancement_time_ms += enhancement_t_ms
            print(f"  Enhancement time: {enhancement_t_ms:.2f} ms")
        
        # --- Inference Block ---
        sahi_result_object = None
        yolo_result_list = None # Standard model output is a list of Results objects
        
        current_inf_block_start_time = time.monotonic()

        print(f"  Running SAHI sliced prediction...")
        sahi_result = get_sliced_prediction(
            image_to_infer_on_bgr, # SAHI expects BGR HWC numpy array
            sahi_detection_model, # This is the AutoDetectionModel instance
            slice_height=SAHI_SLICE_HEIGHT,
            slice_width=SAHI_SLICE_WIDTH,
            overlap_height_ratio=SAHI_OVERLAP_HEIGHT_RATIO,
            overlap_width_ratio=SAHI_OVERLAP_WIDTH_RATIO,
            perform_standard_pred=False, # SAHI_PERFORM_STANDARD_PREDICTION, # Set in Cell 2
            postprocess_type="NMS",
            postprocess_match_metric="IOS",
            postprocess_match_threshold=SAHI_POSTPROCESS_IOU_THRESHOLD,
            postprocess_class_agnostic=False,
            verbose=0
        )
        sahi_result_object = sahi_result
        
        current_inf_block_time_ms = (time.monotonic() - current_inf_block_start_time) * 1000
        total_inference_time_ms += current_inf_block_time_ms # This is the wall-time for the inference block
        print(f"  Total Inference Block time: {current_inf_block_time_ms:.2f} ms")

        # --- Postprocessing & Drawing ---
        annotated_display_frame = original_frame_bgr.copy()
        num_detections_on_frame = 0

        if sahi_result_object.object_prediction_list:
            for pred in sahi_result_object.object_prediction_list:
                if pred.score.value < SAHI_POSTPROCESS_CONF_THRESHOLD: continue
                num_detections_on_frame += 1
                box = pred.bbox; x1,y1,x2,y2 = map(int, box.to_xyxy()); cls_id = pred.category.id; conf = pred.score.value
                label_str = class_names_map.get(cls_id, f"{default_class_label}_{cls_id}")
                cv2.rectangle(annotated_display_frame,(x1,y1),(x2,y2),(255,0,0),2) # Blue for SAHI
                cv2.putText(annotated_display_frame,f"S-{label_str}:{conf:.2f}",(x1,y1-10 if y1>20 else y1+20),cv2.FONT_HERSHEY_SIMPLEX,0.6,(255,0,0),2)
        print(f"  Detections drawn: {num_detections_on_frame}")

        # --- Save and/or Display ---
        if SAVE_ANNOTATED_FRAMES:
            fn_suffix = "_sahi"
            cv2.imwrite(os.path.join(OUTPUT_DIR_RUN, f"frame_{current_original_video_frame_idx:06d}{fn_suffix}.jpg"), annotated_display_frame)
        if DISPLAY_INTERACTIVE_FRAMES:
            title_suffix = "SAHI"
            display_image_jupyter(annotated_display_frame, title=f"{title_suffix} - Ann. Frame #{current_original_video_frame_idx}")
        
        if MAX_FRAMES_TO_PROCESS is not None and frames_actually_processed >= MAX_FRAMES_TO_PROCESS:
            break # Break from inner while True loop
    # End of inner while True loop (frame processing)

    if USE_PYVIDEOREADER and current_pvr_segment_frames_rgb is not None:
        del current_pvr_segment_frames_rgb; gc.collect()
# End of outer for loop (PVR segments / single pass for CV)

# --- Release Resources ---
if USE_PYVIDEOREADER and reader_frame_source: del reader_frame_source; gc.collect()
elif not USE_PYVIDEOREADER and reader_frame_source and reader_frame_source.isOpened(): reader_frame_source.release()
if cap_for_originals.isOpened(): cap_for_originals.release()
gc.collect()

# --- Final Summary ---
print("\n--- Benchmark Run Summary ---")
if frames_actually_processed > 0:
    avg_enhance_ms = (total_enhancement_time_ms / frames_actually_processed) if ENHANCEMENT_LEVEL != 'none' and total_enhancement_time_ms > 0 else 0
    avg_inference_block_ms = total_inference_time_ms / frames_actually_processed
    
    print(f"Total Video Frames (approx): {total_video_frames}")
    print(f"Frames Processed: {frames_actually_processed}")
    if ENHANCEMENT_LEVEL != 'none': print(f"Avg Enhancement Time: {avg_enhance_ms:.2f} ms/frame")
    
    print(f"Avg SAHI Inference Block Time: {avg_inference_block_ms:.2f} ms/frame (includes slicing, multi-inference, NMS)")
    total_avg_ms_cycle = avg_enhance_ms + avg_inference_block_ms

    print(f"Avg Total Cycle Time per Frame: {total_avg_ms_cycle:.2f} ms/frame")
    if total_avg_ms_cycle > 0 : print(f"Equivalent FPS (for processed frames): {1000 / total_avg_ms_cycle:.2f} FPS")
else:
    print("No frames were processed based on the given interval or video length.")

if SAVE_ANNOTATED_FRAMES and os.path.exists(OUTPUT_DIR_RUN) and len(os.listdir(OUTPUT_DIR_RUN)) > 0 :
    print(f"Annotated frames saved to: {OUTPUT_DIR_RUN}")
elif SAVE_ANNOTATED_FRAMES:
    print(f"SAVE_ANNOTATED_FRAMES was True, but no frames seem to have been saved to {OUTPUT_DIR_RUN}.")

print("Notebook execution finished.")


--- Starting Benchmark Processing ---
Video Properties: ~7500 frames, 25.00 FPS (Original)
PVR: Initializing for C:\Users\swest\Desktop\traffic-capturing\sample_videos\5 min test - Large FOV.mp4, target: 640x640
PVR Source: Segments=1

PVR: Decoding segment 1/1 (raw frames 0 to 7499)
PVR: Decoded 7500 frames from segment.
Processing Original Frame #0 (Actual Processed Frame #1)
  Running SAHI sliced prediction...
  Total Inference Block time: 891.00 ms
  Detections drawn: 17
Processing Original Frame #25 (Actual Processed Frame #2)
  Running SAHI sliced prediction...
  Total Inference Block time: 281.00 ms
  Detections drawn: 15
Processing Original Frame #50 (Actual Processed Frame #3)
  Running SAHI sliced prediction...
  Total Inference Block time: 282.00 ms
  Detections drawn: 15
Processing Original Frame #75 (Actual Processed Frame #4)
  Running SAHI sliced prediction...
  Total Inference Block time: 297.00 ms
  Detections drawn: 19
Processing Original Frame #100 (Actual Processed

----------------------------------

-----------------------------------

Using patched_yolo_infer

In [ ]:
# Import the patched_yolo_infer library
from patched_yolo_infer import MakeCropsDetectThem, CombineDetections
from ultralytics import YOLO

# Crop-based inference parameters (equivalent to SAHI)
CROP_HEIGHT = 640  # Equivalent to SAHI_SLICE_HEIGHT
CROP_WIDTH = 640   # Equivalent to SAHI_SLICE_WIDTH
OVERLAP_HEIGHT_PERCENTAGE = 25  # Convert from ratio (0.2) to percentage (20)
OVERLAP_WIDTH_PERCENTAGE = 25   # Convert from ratio (0.2) to percentage (20)
POSTPROCESS_CONF_THRESHOLD = 0.15 # Individual crop confidence threshold
POSTPROCESS_IOU_THRESHOLD = 0.6
NMS_THRESHOLD = 0.8 # Final NMS threshold for combining detections
timestamp_run = datetime.now().strftime("%Y%m%d_%H%M")

print(f"  Crop Params: Crop HxW={CROP_HEIGHT}x{CROP_WIDTH}, Overlap={OVERLAP_HEIGHT_PERCENTAGE}%, Post-Conf={POSTPROCESS_CONF_THRESHOLD}")

# Base Model Confidence for individual crops
BASE_MODEL_CONF_FOR_CROPS = 0.08 # Lower threshold for crop predictions
if CONF_THRESHOLD_OVERRIDE is None or CONF_THRESHOLD_OVERRIDE > BASE_MODEL_CONF_FOR_CROPS:
    CONF_THRESHOLD_OVERRIDE = BASE_MODEL_CONF_FOR_CROPS
    print(f"  Override: CONF_THRESHOLD_OVERRIDE (for base model on crops) set to {CONF_THRESHOLD_OVERRIDE}")

# Advanced crop inference settings
USE_BATCH_INFERENCE = True  # Batch processing for better performance
USE_MEMORY_OPTIMIZE = False  # Memory optimization for segmentation
ENABLE_PROGRESS_TRACKING = False  # Set to True if you want progress bars
CLASS_AGNOSTIC_NMS = False  # True for general detection, False for multi-class scenarios
INTELLIGENT_SORTER = True  # Enable intelligent sorting by area and confidence
SORTER_BINS = 7  # Number of bins for intelligent sorter
MATCH_METRIC = 'IOS'

crop_str_override = "_crops"
reader_str_override = "_pvr" if USE_PYVIDEOREADER and PYVIDEOREADER_AVAILABLE else "_cv"

if os.path.exists(VIDEO_PATH):
    video_name_stem_override = os.path.splitext(os.path.basename(VIDEO_PATH))[0]
else:
    video_name_stem_override = "video_not_found"

# Re-generate OUTPUT_DIR_RUN with crops flag
OUTPUT_DIR_RUN = os.path.join(OUTPUT_DIR_BASE, f"{video_name_stem_override}{reader_str_override}{crop_str_override}_{timestamp_run}")

if SAVE_ANNOTATED_FRAMES:
    os.makedirs(OUTPUT_DIR_RUN, exist_ok=True)
    print(f"  Override: Output directory updated for crops run: {OUTPUT_DIR_RUN}")

print("Crop-based inference setup applied.")
print("-" * 50)

# Use the existing model directly - no wrapper needed
crop_detection_model = model

print("Crop detection model setup complete.")

# Finalize Class Names Map for Annotation
class_names_map_for_annotation = vp.class_id_to_name_map if hasattr(vp, 'class_id_to_name_map') else {}
default_class_label_for_annotation = 'unknown'

if not class_names_map_for_annotation:
    print(f"WARNING: Class name map from VideoProcessor (vp.class_id_to_name_map) is empty.")
    if hasattr(model, 'names') and model.names and isinstance(model.names, dict):
        class_names_map_for_annotation = model.names
        print(f"  Fell back to using model.names for annotation: {class_names_map_for_annotation}")
    else:
        class_names_map_for_annotation = {i: f"{default_class_label_for_annotation}_{i}" for i in range(80)}
        print(f"  CRITICAL FALLBACK: Using very generic names for annotation (e.g., '{default_class_label_for_annotation}_0')")

print(f"\nFinal class name map to be used for annotations: "
      f"{class_names_map_for_annotation if class_names_map_for_annotation else 'ERROR - MAP IS EMPTY'}")

print(f"\n--- Starting Benchmark Processing ---")

# Get Video Properties (Common for both readers)
total_video_frames = 0
original_video_fps = 0.0
_cap_cv_check = cv2.VideoCapture(VIDEO_PATH)
if _cap_cv_check.isOpened():
    total_video_frames = int(_cap_cv_check.get(cv2.CAP_PROP_FRAME_COUNT))
    original_video_fps = _cap_cv_check.get(cv2.CAP_PROP_FPS)
    _cap_cv_check.release()
else:
    print(f"Error: OpenCV could not open video {VIDEO_PATH} to get properties.")
    raise IOError(f"Failed to get video properties for {VIDEO_PATH}")

if total_video_frames == 0 or original_video_fps == 0:
    print("Warning: Could not reliably get video properties. Using defaults.")
    total_video_frames = 1000000
    original_video_fps = TARGET_FPS if TARGET_FPS > 0 else 25.0

print(f"Video Properties: ~{total_video_frames} frames, {original_video_fps:.2f} FPS (Original)")
frame_skip_interval = FRAME_INTERVAL_CONFIG

# Initialize Counters
frames_actually_processed = 0
total_preprocessing_time_ms_std = 0
total_inference_time_ms = 0
total_enhancement_time_ms = 0
total_nms_combination_time_ms = 0
raw_frame_index_overall = 0

# OpenCV VideoCapture for getting original frames for annotation
cap_for_originals = cv2.VideoCapture(VIDEO_PATH)
if not cap_for_originals.isOpened():
    print(f"CRITICAL ERROR: OpenCV could not open video {VIDEO_PATH} for original frames.")
    raise IOError(f"Failed to open video for original frames: {VIDEO_PATH}")

# Video Reading Loop Setup
reader_frame_source = None
segment_iterator = range(1)
pvr_segment_details = {}

if USE_PYVIDEOREADER:
    print(f"PVR: Initializing for {VIDEO_PATH}, target: {MODEL_INPUT_SIZE}x{MODEL_INPUT_SIZE}")
    reader_frame_source = PyVideoReader(VIDEO_PATH, device='cpu',
                                        resize_shorter_side=MODEL_INPUT_SIZE,
                                        resize_longer_side=MODEL_INPUT_SIZE)
    pvr_segment_details['num_segments'] = math.ceil(total_video_frames / PYVIDEOREADER_DECODE_SEGMENT_SIZE)
    segment_iterator = range(pvr_segment_details['num_segments'])
    print(f"PVR Source: Segments={pvr_segment_details['num_segments']}")
else:
    print("OpenCV Source")
    reader_frame_source = cv2.VideoCapture(VIDEO_PATH)
    if not reader_frame_source.isOpened():
        raise IOError(f"OpenCV reader failed for {VIDEO_PATH}")

# Main Loop
for seg_idx in segment_iterator:
    if MAX_FRAMES_TO_PROCESS is not None and frames_actually_processed >= MAX_FRAMES_TO_PROCESS:
        break

    current_pvr_segment_frames_rgb = None
    pvr_current_segment_start_idx = 0

    if USE_PYVIDEOREADER:
        pvr_current_segment_start_idx = seg_idx * PYVIDEOREADER_DECODE_SEGMENT_SIZE
        segment_end_raw_idx = min((seg_idx + 1) * PYVIDEOREADER_DECODE_SEGMENT_SIZE, total_video_frames)
        if pvr_current_segment_start_idx >= segment_end_raw_idx:
            continue
        
        print(f"\nPVR: Decoding segment {seg_idx+1}/{pvr_segment_details['num_segments']} "
              f"(raw frames {pvr_current_segment_start_idx} to {segment_end_raw_idx-1})")
        current_pvr_segment_frames_rgb = reader_frame_source.decode(start_frame=pvr_current_segment_start_idx,
                                                                    end_frame=segment_end_raw_idx)
        if current_pvr_segment_frames_rgb is None or current_pvr_segment_frames_rgb.shape[0] == 0:
            print(f"PVR: Segment {seg_idx+1} returned no frames.")
            continue
        print(f"PVR: Decoded {current_pvr_segment_frames_rgb.shape[0]} frames from segment.")
    
    frame_idx_in_current_pvr_segment = -1

    while True:
        if MAX_FRAMES_TO_PROCESS is not None and frames_actually_processed >= MAX_FRAMES_TO_PROCESS:
            break
        
        # Get frame content for model and its original index
        current_original_video_frame_idx = -1
        is_input_rgb = False

        if USE_PYVIDEOREADER:
            frame_idx_in_current_pvr_segment += 1
            if frame_idx_in_current_pvr_segment >= current_pvr_segment_frames_rgb.shape[0]:
                break
            current_original_video_frame_idx = pvr_current_segment_start_idx + frame_idx_in_current_pvr_segment
            is_input_rgb = True 
        else:
            ret_cv, frame_bgr_original_size = reader_frame_source.read()
            if not ret_cv:
                break
            current_original_video_frame_idx = int(reader_frame_source.get(cv2.CAP_PROP_POS_FRAMES)) - 1
            is_input_rgb = False

        raw_frame_index_overall = current_original_video_frame_idx

        if current_original_video_frame_idx % frame_skip_interval != 0:
            continue

        # Get original BGR frame for drawing and for crop-based input
        original_frame_bgr = None
        cap_for_originals.set(cv2.CAP_PROP_POS_FRAMES, current_original_video_frame_idx)
        ret_orig, original_frame_bgr = cap_for_originals.read()
        if not ret_orig or original_frame_bgr is None:
            print(f"Warning: OpenCV could not read original frame at index {current_original_video_frame_idx} for drawing.")
            continue

        frames_actually_processed += 1
        print(f"Processing Original Frame #{current_original_video_frame_idx} (Actual Processed Frame #{frames_actually_processed})")

        # Image for Inference (BGR, HWC, Original Size, possibly enhanced)
        image_to_infer_on_bgr = original_frame_bgr.copy()

        if ENHANCEMENT_LEVEL and ENHANCEMENT_LEVEL != 'none':
            start_enhance_t = time.monotonic()
            img_rgb_to_enhance = cv2.cvtColor(image_to_infer_on_bgr, cv2.COLOR_BGR2RGB)
            enhanced_rgb = enhance_frame_for_detection(img_rgb_to_enhance, ENHANCEMENT_LEVEL)
            image_to_infer_on_bgr = cv2.cvtColor(enhanced_rgb, cv2.COLOR_RGB2BGR)
            enhancement_t_ms = (time.monotonic() - start_enhance_t) * 1000
            total_enhancement_time_ms += enhancement_t_ms
            print(f"  Enhancement time: {enhancement_t_ms:.2f} ms")
        
        # Inference Block - Crop-based inference with intelligent NMS
        current_inf_block_start_time = time.monotonic()

        print(f"  Running crop-based prediction...")
        
        try:
            # Create MakeCropsDetectThem instance for this frame
            element_crops = MakeCropsDetectThem(
                image=image_to_infer_on_bgr,  # Expects BGR format
                model=crop_detection_model,
                imgsz=MODEL_INPUT_SIZE,
                conf=BASE_MODEL_CONF_FOR_CROPS,
                iou=POSTPROCESS_IOU_THRESHOLD,
                classes_list=None,  # Process all classes, can be set to specific classes if needed
                segment=False,  # Set to True if you need segmentation
                shape_x=CROP_WIDTH,
                shape_y=CROP_HEIGHT,
                overlap_x=OVERLAP_WIDTH_PERCENTAGE,
                overlap_y=OVERLAP_HEIGHT_PERCENTAGE,
                show_crops=False,
                show_processing_status=ENABLE_PROGRESS_TRACKING,
                resize_initial_size=True,
                memory_optimize=USE_MEMORY_OPTIMIZE,
                batch_inference=USE_BATCH_INFERENCE
            )
            
            print(f"    Generated {len(element_crops.crops)} crops from frame")
            
            # Combine detections using intelligent NMS
            nms_start_time = time.monotonic()
            
            result = CombineDetections(
                element_crops=element_crops,
                nms_threshold=NMS_THRESHOLD,
                match_metric=MATCH_METRIC,  # 'IOS' or 'IOU' - IOS generally works better for overlapping crops
                class_agnostic_nms=CLASS_AGNOSTIC_NMS,
                intelligent_sorter=INTELLIGENT_SORTER,
                sorter_bins=SORTER_BINS
            )
            
            nms_time_ms = (time.monotonic() - nms_start_time) * 1000
            total_nms_combination_time_ms += nms_time_ms
            
        except Exception as e:
            print(f"  ERROR during crop-based inference: {e}")
            continue
        
        current_inf_block_time_ms = (time.monotonic() - current_inf_block_start_time) * 1000
        total_inference_time_ms += current_inf_block_time_ms
        print(f"  Total Inference Block time: {current_inf_block_time_ms:.2f} ms (Crops: {current_inf_block_time_ms-nms_time_ms:.2f} ms, NMS: {nms_time_ms:.2f} ms)")

        # Postprocessing & Drawing - Use filtered results from CombineDetections
        annotated_display_frame = original_frame_bgr.copy()
        num_detections_on_frame = 0
        
        # Access the filtered results from CombineDetections
        if hasattr(result, 'filtered_boxes') and len(result.filtered_boxes) > 0:
            for i in range(len(result.filtered_boxes)):
                conf = result.filtered_confidences[i]
                if conf < POSTPROCESS_CONF_THRESHOLD:
                    continue
                
                num_detections_on_frame += 1
                x1, y1, x2, y2 = map(int, result.filtered_boxes[i])
                cls_id = result.filtered_classes_id[i]
                
                # Get class name from the filtered results or fall back to map
                if hasattr(result, 'filtered_classes_names') and i < len(result.filtered_classes_names):
                    label_str = result.filtered_classes_names[i]
                else:
                    label_str = class_names_map_for_annotation.get(cls_id, f"{default_class_label_for_annotation}_{cls_id}")
                
                # Draw bounding box and label
                cv2.rectangle(annotated_display_frame, (x1, y1), (x2, y2), (0, 255, 0), 2)  # Green for crops
                cv2.putText(annotated_display_frame, f"C-{label_str}:{conf:.2f}", 
                           (x1, y1-10 if y1>20 else y1+20), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

        print(f"  Detections drawn: {num_detections_on_frame}")

        # Optional: Print detailed patch information
        if ENABLE_PROGRESS_TRACKING:
            element_crops.patches_info()

        # Save and/or Display
        if SAVE_ANNOTATED_FRAMES:
            fn_suffix = "_crops"
            cv2.imwrite(os.path.join(OUTPUT_DIR_RUN, f"frame_{current_original_video_frame_idx:06d}{fn_suffix}.jpg"), annotated_display_frame)
        if DISPLAY_INTERACTIVE_FRAMES:
            title_suffix = "Crops"
            display_image_jupyter(annotated_display_frame, title=f"{title_suffix} - Ann. Frame #{current_original_video_frame_idx}")
        
        # Clean up to free memory
        del element_crops, result
        gc.collect()
        
        if MAX_FRAMES_TO_PROCESS is not None and frames_actually_processed >= MAX_FRAMES_TO_PROCESS:
            break

    if USE_PYVIDEOREADER and current_pvr_segment_frames_rgb is not None:
        del current_pvr_segment_frames_rgb
        gc.collect()

# Release Resources
if USE_PYVIDEOREADER and reader_frame_source: 
    del reader_frame_source
    gc.collect()
elif not USE_PYVIDEOREADER and reader_frame_source and reader_frame_source.isOpened(): 
    reader_frame_source.release()
if cap_for_originals.isOpened(): 
    cap_for_originals.release()
gc.collect()

# Final Summary
print("\n--- Benchmark Run Summary ---")
if frames_actually_processed > 0:
    avg_enhance_ms = (total_enhancement_time_ms / frames_actually_processed) if ENHANCEMENT_LEVEL != 'none' and total_enhancement_time_ms > 0 else 0
    avg_inference_block_ms = total_inference_time_ms / frames_actually_processed
    avg_nms_ms = total_nms_combination_time_ms / frames_actually_processed
    avg_crop_inference_ms = avg_inference_block_ms - avg_nms_ms
    
    print(f"Total Video Frames (approx): {total_video_frames}")
    print(f"Frames Processed: {frames_actually_processed}")
    if ENHANCEMENT_LEVEL != 'none': 
        print(f"Avg Enhancement Time: {avg_enhance_ms:.2f} ms/frame")
    
    print(f"Avg Crop Inference Time: {avg_crop_inference_ms:.2f} ms/frame")
    print(f"Avg NMS Combination Time: {avg_nms_ms:.2f} ms/frame")
    print(f"Avg Total Inference Block Time: {avg_inference_block_ms:.2f} ms/frame")
    total_avg_ms_cycle = avg_enhance_ms + avg_inference_block_ms

    print(f"Avg Total Cycle Time per Frame: {total_avg_ms_cycle:.2f} ms/frame")
    if total_avg_ms_cycle > 0: 
        print(f"Equivalent FPS (for processed frames): {1000 / total_avg_ms_cycle:.2f} FPS")
        
    # Performance insights
    print(f"\nPerformance Breakdown:")
    if avg_enhance_ms > 0:
        print(f"  Enhancement: {(avg_enhance_ms/total_avg_ms_cycle)*100:.1f}%")
    print(f"  Crop Inference: {(avg_crop_inference_ms/total_avg_ms_cycle)*100:.1f}%")
    print(f"  NMS Combination: {(avg_nms_ms/total_avg_ms_cycle)*100:.1f}%")
else:
    print("No frames were processed based on the given interval or video length.")

if SAVE_ANNOTATED_FRAMES and os.path.exists(OUTPUT_DIR_RUN) and len(os.listdir(OUTPUT_DIR_RUN)) > 0:
    print(f"Annotated frames saved to: {OUTPUT_DIR_RUN}")
elif SAVE_ANNOTATED_FRAMES:
    print(f"SAVE_ANNOTATED_FRAMES was True, but no frames seem to have been saved to {OUTPUT_DIR_RUN}.")

print("Notebook execution finished.")


  Crop Params: Crop HxW=640x640, Overlap=25%, Post-Conf=0.15
  Override: Output directory updated for crops run: C:\Users\swest\Desktop\traffic-capturing\traffic-reporting\utility_scripts\benchmark_detection_outputs\5 min test - Large FOV_pvr_crops_20250604_0105
Crop-based inference setup applied.
--------------------------------------------------
Crop detection model setup complete.

Final class name map to be used for annotations: {0: 'Bus', 1: 'Cyclist', 2: 'Light Vehicle', 3: 'Long Truck', 4: 'Medium Truck', 5: 'Minibus Taxi', 6: 'Motorcycle', 7: 'Pedestrian', 8: 'Short Truck'}

--- Starting Benchmark Processing ---
Video Properties: ~7500 frames, 25.00 FPS (Original)
PVR: Initializing for C:\Users\swest\Desktop\traffic-capturing\sample_videos\5 min test - Large FOV.mp4, target: 640x640
PVR Source: Segments=1

PVR: Decoding segment 1/1 (raw frames 0 to 7499)
PVR: Decoded 7500 frames from segment.
Processing Original Frame #0 (Actual Processed Frame #1)
  Running crop-based predicti